# Gateway IAMロールInboundでAWS LambdaをMCP化する
## Bedrock AgentCore GatewayでAWS Lambda関数を安全なMCPツールに変換する

## 概要
Bedrock AgentCore Gatewayは、既存のAWS Lambda関数を完全管理型のMCPサーバーに変換する方法を提供し、インフラやホスティングの管理を不要にします。Gatewayは、これらすべてのツールにわたって統一されたModel Context Protocol（MCP）インターフェースを提供します。Gatewayは、着信リクエストとターゲットリソースへのアウトバウンド接続の両方に対して安全なアクセス制御を確保するために、デュアル認証モデルを採用しています。フレームワークは2つの主要コンポーネントで構成されています：Gatewayターゲットへのアクセスを試みるユーザーを検証および承認するInbound Authと、認証されたユーザーに代わってGatewayがバックエンドリソースに安全に接続できるようにするOutbound Authです。Gatewayは、アウトバウンド認証のためにAWS Lambda関数への呼び出しを承認するためにIAMロールを使用します。

この例では、IAMロールを使用したインバウンド認証とアウトバウンド認証の両方を実演します。

![動作の仕組み](images/lambda-gw-iam-inbound.png)

### チュートリアルの詳細


| 情報              | 詳細                                                       |
|:------------------|:-----------------------------------------------------------|
| チュートリアルタイプ | インタラクティブ                                           |
| AgentCoreコンポーネント | AgentCore Gateway                                         |
| エージェントフレームワーク | Strands Agents                                            |
| Gatewayターゲットタイプ | AWS Lambda                                                |
| Inbound Auth      | AWS IAM                                                    |
| Outbound Auth     | AWS IAM                                                   |
| LLMモデル         | Anthropic Claude Haiku 4.5、Amazon Nova Pro              |
| チュートリアルコンポーネント | AgentCore Gatewayの作成とAgentCore Gatewayの呼び出し      |
| チュートリアル垂直領域 | クロス垂直領域                                             |
| 例の複雑さ        | 簡単                                                       |
| 使用SDK           | boto3                                                     |

チュートリアルの最初の部分では、Lambda用のAmazonCore Gatewayターゲットを作成します

### チュートリアルアーキテクチャ
このチュートリアルでは、AWS Lambda関数で定義された操作をMCPツールに変換し、Bedrock AgentCore Gatewayでホストします。AWS Sigv4ヘッダーでAWS IAM認証情報を使用したイングレス認証を実演します。
デモンストレーションの目的で、Amazon Bedrockモデルを使用するStrands Agentを使用します。
この例では、2つのツール（get_orderとupdate_order）を持つ非常にシンプルなエージェントを使用します。

## 前提条件

このチュートリアルを実行するには、以下が必要です：
* Jupyterノートブック（Pythonカーネル）
* uv
* AWS認証情報
* AWSコンソール経由でのNova Proへのアクセス
* Amazon Bedrock AgentCore SDK
* Strand Agents

## 着信AgentCore Gatewayリクエストの認証設定
AgentCore Gatewayは、インバウンド認証とアウトバウンド認証を介して安全な接続を提供します。インバウンド認証の場合、AgentCore Gatewayは、Gatewayを呼び出すためにOAuthに加えてAWS IAM認証情報/アイデンティティをサポートしています。ツールが外部リソースへのアクセスを必要とする場合、AgentCore GatewayはAPIキー、IAM、またはOAuthトークンを介したアウトバウンド認証を使用して、外部リソースへのアクセスを許可または拒否できます。

インバウンド認証フロー中、エージェントまたはMCPクライアントは、認証に使用され、AgentCore GatewayにアクセスするためのIAM権限に対して承認されるAWS Signature V4署名付きリクエストを使用します。AgentCore Gatewayは、AWS IAM認証情報/アイデンティティを検証し、インバウンド認証を実行します。

AgentCore Gatewayで実行されているツールが外部リソースへのアクセスを必要とする場合、IAMロールはGatewayターゲットのダウンストリームリソースの認証情報を取得します。AgentCore Gatewayは、ダウンストリームAPIへのアクセスを得るために、認証情報を呼び出し元に渡します。

In [ ]:
!pip install --force-reinstall -U -r requirements.txt --quiet

In [ ]:
# Amazon SageMakerノートブックを使用していない場合はAWS認証情報を設定
import os

#os.environ['AWS_ACCESS_KEY_ID']=''
#os.environ['AWS_SECRET_ACCESS_KEY']=''
os.environ['AWS_DEFAULT_REGION']='us-west-2' # AWSリージョンを設定




In [ ]:
import os
import sys

# 現在のスクリプトのディレクトリを取得
if '__file__' in globals():
    current_dir = os.path.dirname(os.path.abspath(__file__))
else:
    current_dir = os.getcwd()  # __file__が定義されていない場合のフォールバック（例：Jupyter）

# utils.pyを含むディレクトリに移動（1レベル上）
utils_dir = os.path.abspath(os.path.join(current_dir, '..'))

# sys.pathに追加
sys.path.insert(0, utils_dir)



In [ ]:
# これでutilsをインポートできます
import utils
#### MCPツールに変換したいサンプルAWS Lambda関数を作成
lambda_resp = utils.create_gateway_lambda("lambda_function_code.zip")
if lambda_resp is not None:
    if lambda_resp['exit_code'] == 0:
        print("Lambda関数がARNで作成されました: ", lambda_resp['lambda_function_arn'])
    else:
        print("Lambda関数の作成が失敗しました。メッセージ: ", lambda_resp['lambda_function_arn'])

In [ ]:
#### Gatewayが引き受けるIAMロールを作成
import utils
agentcore_gateway_iam_role = utils.create_agentcore_gateway_role("sample-lambdagateway")
print("Agentcore gatewayロールARN: ", agentcore_gateway_iam_role['Role']['Arn'])

# インバウンド認証用のAmazon IAM AuthorizerでGatewayを作成

In [ ]:
import time
import boto3
# Amazon IAMでGatewayを作成。
gateway_client = boto3.client('bedrock-agentcore-control', region_name = os.environ['AWS_DEFAULT_REGION'])

create_response = gateway_client.create_gateway(name='TestGWforLambdaIAM',
    roleArn = agentcore_gateway_iam_role['Role']['Arn'], # IAMロールはGatewayの作成/リスト/取得/削除の権限を持っている必要があります
    protocolType='MCP',
    authorizerType='AWS_IAM',
    description='イングレス認証にAmazon IAMを使用したAWS LambdaターゲットタイプのAgentCore Gateway'
)
print(create_response)
# GatewayTarget作成に使用されるGatewayIDを取得
gatewayID = create_response["gatewayId"]
gatewayURL = create_response["gatewayUrl"]
print(gatewayID)
time.sleep(10)

# AWS Lambdaターゲットを作成し、MCPツールに変換

In [ ]:
# 以下のAWS Lambda関数ARNを置き換えてください
lambda_target_config = {
    "mcp": {
        "lambda": {
            "lambdaArn": lambda_resp['lambda_function_arn'], # これをあなたのAWS Lambda関数ARNに置き換えてください
            "toolSchema": {
                "inlinePayload": [
                    {
                        "name": "get_order_tool",
                        "description": "注文を取得するツール",
                        "inputSchema": {
                            "type": "object",
                            "properties": {
                                "orderId": {
                                    "type": "string"
                                }
                            },
                            "required": ["orderId"]
                        }
                    },                    
                    {
                        "name": "update_order_tool",
                        "description": "orderIdを更新するツール",
                        "inputSchema": {
                            "type": "object",
                            "properties": {
                                "orderId": {
                                    "type": "string"
                                }
                            },
                            "required": ["orderId"]
                        }
                    }
                ]
            }
        }
    }
}

credential_config = [ 
    {
        "credentialProviderType" : "GATEWAY_IAM_ROLE"
    }
]
targetname='LambdaUsingSDK'
response = gateway_client.create_gateway_target(
    gatewayIdentifier=gatewayID,
    name=targetname,
    description='SDKを使用したLambdaターゲット',
    targetConfiguration=lambda_target_config,
    credentialProviderConfigurations=credential_config)

# Gatewayを呼び出すためのAWS IAMロールを作成 

以下の関数は、AWS Bedrock AgentCoreが指定されたGatewayを呼び出すことを許可するIAMロールを作成または更新します。指定されたGateway IDに対してbedrock-agentcore:InvokeGateway権限を付与するインラインポリシーを構築してアタッチします。また、Bedrock AgentCoreサービスと呼び出し元のIAMエンティティ（current_arn）の両方がロールを引き受けることを許可する信頼ポリシーを設定します。

In [ ]:
#### Gatewayを呼び出すIAMロールを作成
current_role_arn = utils.get_current_role_arn()
print("現在のロールARN: ", current_role_arn)

agentcore_gateway_iam_invoke_role = utils.create_gateway_invoke_tool_role("gateway-invoke-role",gatewayID , current_role_arn)
print("Agentcore gatewayを呼び出すロールARN: ", agentcore_gateway_iam_invoke_role['Role']['Arn'])

# Bedrock AgentCore Gatewayを使用してAWS LambdaのMCPツールを呼び出すStrandsエージェント

#### MCPクライアントSDKでのAWS IAM認証のサポート

AgentCore GatewayへのインバウンドリクエストでAWS IAM認証がサポートされるようになりましたが、現在のオープンソースMCPクライアントSDKは、特にストリーミングHTTP接続でのSigV4認証のサポートが限られていることに注意することが重要です。ただし、AWSは「AWS LambdaでModel Context Protocol（MCP）サーバーを実行する」プロジェクトを通じてソリューションを提供しており、ストリーミングHTTP接続でのSigV4認証のための重要な拡張機能が含まれています。

[AWS Labs GitHubリポジトリ](https://github.com/awslabs/run-model-context-protocol-servers-with-aws-lambda/tree/main)で利用可能なこの実装は、ストリーミング接続の認証ギャップを埋め、StrandsやLangChainなどの人気のあるエージェントフレームワークとシームレスに統合できます。StreamableHTTPTransportWithSigV4クラスは、標準のMCPトランスポートレイヤーを拡張して、ストリーミング機能を維持しながらAWS SigV4署名を処理し、AgentCore Gatewayの新しいIAM認証機能と互換性を持たせます。

In [ ]:
!pip3 install --upgrade strands-agents strands-agents-tools
from strands.models import BedrockModel

## ~/.aws/credentialsに設定されたIAM認証情報はBedrockモデルへのアクセス権限を持っている必要があります
yourmodel = BedrockModel(
    model_id="us.amazon.nova-pro-v1:0",
    temperature=0.7,
)

In [ ]:
from strands import Agent
import logging
from strands import Agent
import logging
from strands.tools.mcp.mcp_client import MCPClient
from mcp.client.streamable_http import streamablehttp_client 
from botocore.credentials import Credentials
from streamable_http_sigv4 import (
    streamablehttp_client_with_sigv4,
)

SERVICE="bedrock-agentcore"

# ルートstrandsロガーを設定。問題をデバッグしている場合はDEBUGに変更してください。
logging.getLogger("strands").setLevel(logging.INFO)

# ログを表示するためのハンドラーを追加
logging.basicConfig(
    format="%(levelname)s | %(name)s | %(message)s", 
    handlers=[logging.StreamHandler()]
)

def create_streamable_http_transport(mcp_url: str, access_token: str):
       return streamablehttp_client(mcp_url, headers={"Authorization": f"Bearer {access_token}"})

def create_streamable_http_transport_sigv4(mcp_url: str, key: str, secret: str, sessionToken: str, serviceName: str, awsRegion: str):
        iamcredentials = Credentials(
            access_key=key,
            secret_key=secret,
            token = sessionToken
        )
        return streamablehttp_client_with_sigv4(
            url=mcp_url,
            credentials=iamcredentials,
            service=serviceName,
            region=awsRegion,
        )

def get_full_tools_list(client):
    more_tools = True
    tools = []
    pagination_token = None
    while more_tools:
        tmp_tools = client.list_tools_sync(pagination_token=pagination_token)
        tools.extend(tmp_tools)
        if tmp_tools.pagination_token is None:
            more_tools = False
        else:
            more_tools = True 
            pagination_token = tmp_tools.pagination_token
    return tools

def call_tool_sync(client, tool_id,tool_name, parameters=None):
    # ツールを呼び出す（ページネーション引数はサポートされていません）
    response = client.call_tool_sync(
        tool_use_id=tool_id,
        name=tool_name,
        arguments=parameters
    )

    # 出力コンテンツを抽出
    if hasattr(response, "results") and response.results:
        return response.results
    elif hasattr(response, "output") and response.output:
        return response.output
    elif hasattr(response, "content"):
        return response.content
    else:
        return response  # フォールバック
         
def run_agent(mcp_url: str,key: str, secret: str, sessionToken: str,serviceName: str, awsRegion: str):
    mcp_client = MCPClient(lambda: create_streamable_http_transport_sigv4(mcp_url,key,secret,sessionToken,serviceName, awsRegion))

    with mcp_client:
        tools = get_full_tools_list(mcp_client)
        print(f"以下のツールが見つかりました: {[tool.tool_name for tool in tools]}")
        print(f"ツール名: {tools[0].tool_name}")
        
        
        agent = Agent(model=yourmodel,tools=tools) ## お好みのモデルに置き換えることができます
        print(f"エージェントに読み込まれたツールは {agent.tool_names}")
        agent("注文ID 123の注文ステータスを確認し、ツールからの正確なレスポンスを表示してください")
        # ツールでmcpを呼び出す
        tool=tools[0].tool_name
        tool_id="get-order-id-123-call-1"
        result = call_tool_sync(
            mcp_client,
            tool_id,
            tool_name=tool,
            parameters={"orderId": "123"}
        )

        print(f"ツール呼び出し結果: {result['content'][0]['text']}")
        

IAM Gateway Invokeロールを引き受け、エージェントを実行

In [ ]:
sts_client = boto3.client("sts")
response = sts_client.assume_role(
        RoleArn=agentcore_gateway_iam_invoke_role['Role']['Arn'],
        RoleSessionName="invoke_mcp_session",
        DurationSeconds=3600  # 1時間、一部のロールでは最大12時間まで可能
)

creds = response["Credentials"]

access=creds["AccessKeyId"]
secret=creds["SecretAccessKey"]
token=creds["SessionToken"]

# この新しいgateway-invoke-roleの認証情報でエージェントを実行
time.sleep(10) 
run_agent(gatewayURL,access,secret,token, SERVICE, os.environ['AWS_DEFAULT_REGION'])

**問題: 以下のセルを実行中に以下のエラーが発生した場合、pydanticとpydantic-coreのバージョン間の非互換性を示しています。**

```
TypeError: model_schema() got an unexpected keyword argument 'generic_origin'
```
**解決方法**

pydantic==2.7.2とpydantic-core 2.27.2の両方が互換性があることを確認する必要があります。完了したらカーネルを再起動してください。

# クリーンアップ

IAMロール、IAMポリシー、認証情報プロバイダー、AWS Lambda関数などの追加リソースも作成されており、クリーンアップの一部として手動で削除する必要がある場合があります。これは実行する例によって異なります。

## Gatewayを削除（オプション）

In [ ]:
import utils
utils.delete_gateway(gateway_client,gatewayID)